# Phishing Website Detection: demo notebook

Machine Learning course project. This notebook demonstrates the whole project using its real code (`src/`):

1. **Setup**: load the project and install the libraries
2. **Dataset**: what the data looks like
3. **Model results**: four models on the full (30) and lite (21) feature sets
4. **Feature importance**: which features matter (permutation, SHAP, ablation)
5. **How XGBoost decides**: inside the served model
6. **Live demo**: check real URLs and a fake phishing page, with explanations

The trained models in `models/` are used as they are, so the demo takes a few minutes. Set `RETRAIN = True` in step 1c to retrain everything from scratch (about 15-20 min on Colab's CPU).

## 1. Setup

**1a. Upload the project.** In Colab, run the next cell and choose `phishing_project.zip`. Run locally (Jupyter in the project folder), it skips the upload.

In [ ]:
import os, sys, zipfile

IN_COLAB = "google.colab" in sys.modules
PROJECT = "/content/phishing_project" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.exists(PROJECT):
    from google.colab import files
    uploaded = files.upload()                      # choose phishing_project.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content")
print("Project folder:", PROJECT)
print(sorted(os.listdir(PROJECT)))

**1b. Install the pinned library versions** (Colab only, about 1-2 min). The saved models need these exact versions of scikit-learn and XGBoost.

⚠️ After this cell finishes, Colab may say the session needs a restart (because numpy/pandas were upgraded). If so, click **Runtime → Restart session**, then continue from **1c**. Your uploaded files are kept.

In [ ]:
import subprocess
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{PROJECT}/requirements.txt"], check=True)
    print("Installed. If Colab asks you to restart the session, do it and continue from 1c.")
else:
    print("Running locally: using the existing environment.")

**1c. Start here after a restart.** Move into the project folder and check the library versions.

In [ ]:
import os, sys
PROJECT = "/content/phishing_project" if os.path.exists("/content/phishing_project") else os.getcwd()
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

import warnings
warnings.filterwarnings("ignore")

import sklearn, xgboost, shap, pandas as pd, numpy as np
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__,
      "| shap", shap.__version__, "| pandas", pd.__version__)
assert sklearn.__version__ == "1.9.1" and xgboost.__version__ == "3.4.1", \
    "Wrong library versions: run cell 1b and restart the session."

RETRAIN = False   # True = retrain all 8 models and redo every analysis (~15-20 min on Colab)
if RETRAIN:
    import subprocess
    subprocess.run([sys.executable, "-m", "src.run_all"], check=True)

## 2. The dataset

UCI Phishing Websites dataset: 11,055 websites, 30 features. Each feature is coded **-1** (phishing-like), **0** (suspicious) or **1** (legitimate-like). We map the label to **phishing = 1**, legitimate = 0.

In [ ]:
from src import data
from src.features import FULL, LITE, EXCLUDED_FROM_LITE

X, y = data.load()
print("Shape:", X.shape)
print(f"Phishing: {y.sum()}  Legitimate: {(1 - y).sum()}  ({y.mean():.1%} phishing)")
X.head()

In [ ]:
import json
summary = json.load(open("results/data_summary.json"))
print(f"Exact duplicate rows: {summary['duplicate_rows']} of {summary['rows']} "
      f"({summary['duplicate_rows'] / summary['rows']:.0%}); unique rows: {summary['unique_rows']}")

X_tr, X_te, y_tr, y_te = data.split(X, y)
print(f"Train: {len(X_tr)} rows   Test: {len(X_te)} rows (stratified 80/20, seed 42)")
print(f"\nLite set: {len(LITE)} features the live extractor can compute.")
print("Left out (need WHOIS, discontinued services, search engines or blacklists):")
print("  " + ", ".join(EXCLUDED_FROM_LITE))

In [ ]:
# Share of phishing sites for each value of the strongest features
counts = pd.read_csv("results/feature_value_counts.csv")
counts[counts.feature.isin(["sslfinal_state", "url_of_anchor", "web_traffic", "prefix_suffix"])]

## 3. Model results

Four models, each tuned with **5-fold cross-validation** on the training set (scored on F1), then tested **once** on the held-out 20%.

In [ ]:
cols = ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "cv_f1_mean", "cv_f1_std",
        "train_time_s", "inference_ms"]
for fs in ("full", "lite"):
    df = pd.read_csv(f"results/metrics_{fs}.csv")[cols].set_index("model")
    print(f"\n=== {fs.upper()} feature set ({30 if fs == 'full' else 21} features), test set ===")
    display(df.round(4))

In [ ]:
from IPython.display import Image, display
for f in ["roc_full.png", "confusion_full.png", "roc_lite.png", "confusion_lite.png"]:
    display(Image(f"results/{f}"))

### Sensitivity check: duplicate rows

47% of the rows are duplicates, so some test rows are exact copies of training rows. Here every model is retrained and retested **with the duplicates removed**: a harder, more honest test.

In [ ]:
dedup = pd.read_csv("results/dedup_sensitivity.csv")
dedup[["feature_set", "model", "accuracy_standard", "accuracy_dedup", "f1_standard", "f1_dedup", "f1_change"]].round(4)

In [ ]:
serving = json.load(open("models/serving.json"))
print("Model served by the demo:", serving["model"])
print("Why:", serving["reason"])

## 4. Feature importance

Three kinds of importance: **built-in** (trees), **permutation** (how much test F1 drops when a feature is shuffled; works for every model) and **SHAP** (each feature's contribution to each prediction).

In [ ]:
for f in ["importance_permutation.png", "shap_summary.png", "importance_agreement.png"]:
    display(Image(f"results/{f}"))
ranks = pd.read_csv("results/importance_ranks.csv", index_col=0)
print("Consensus top 10 (mean rank over all methods):", list(ranks.index[:10]))

**Ablation:** how many features are really needed? The Random Forest is retrained on its top-k features (ranked with SHAP on training data).

In [ ]:
display(Image("results/ablation.png"))
pd.read_csv("results/ablation.csv")[["k", "cv_accuracy", "cv_f1", "test_accuracy"]]

**Full vs lite:** the lite set is about 3 points worse. Which missing features explain the gap? Each excluded feature is added back to the lite set, alone and then cumulatively (cross-validation on the training set).

In [ ]:
display(Image("results/lite_gap.png"))
gap = pd.read_csv("results/lite_gap.csv")
gap["share_of_gap_recovered"] = gap["share_of_gap_recovered"].map("{:.0%}".format)
gap[["feature_set", "added", "cv_f1", "share_of_gap_recovered"]].round(4)

## 5. Inside the served XGBoost model

XGBoost adds up the scores of many small decision trees, each built to fix the errors of the previous ones. The total is a log-odds score, which a sigmoid turns into a phishing probability.

In [ ]:
import joblib, shap
model = joblib.load(f"models/{serving['model_file']}")
explainer = shap.TreeExplainer(model)

print("Hyper-parameters:", {k: model.get_params()[k] for k in
      ["n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree"]})
base = float(np.ravel(explainer.expected_value)[0])
print(f"Starting score (base value): {base:.3f} log-odds = {1 / (1 + np.exp(-base)):.1%} phishing "
      "-- the share of phishing sites in the training data")

# The first tree, as text
print("\nFirst tree (first lines):")
print("\n".join(model.get_booster().get_dump()[0].splitlines()[:12]))

## 6. Live demo

`check(url)` does exactly what the API does:
1. compute the 21 lite features from the URL and the page's static HTML (`src/extract.py`)
2. score them with the XGBoost model
3. explain the result with per-prediction SHAP values (positive = pushes towards phishing)

In [ ]:
import matplotlib.pyplot as plt
from src.extract import extract, to_vector
from src.features import DESCRIPTIONS


def check(url, html=None, plot=True):
    """Classify a URL. With `html`, that HTML is used instead of downloading the page."""
    if html is None:
        r = extract(url)
    else:
        r = extract(url, resolver=lambda host: True,
                    fetcher=lambda u: {"html": html, "final_url": u, "n_redirects": 0,
                                       "cert_valid": True, "is_html": True})
    vec = to_vector(r, serving["fill_values"])
    row = pd.DataFrame([vec], columns=LITE)
    p = float(model.predict_proba(row)[0, 1])
    contrib = explainer(row).values[0]

    verdict = "PHISHING" if p >= 0.5 else "LEGITIMATE"
    print(f"{url}\n  -> {verdict}   (phishing probability {p:.2%})")
    if r["partial"]:
        print("  ! partial result:", "; ".join(r["errors"]))

    table = pd.DataFrame({"feature": LITE, "checks": [DESCRIPTIONS[f] for f in LITE],
                          "value": vec, "shap": contrib, "computed": [r["available"][f] for f in LITE]})
    table = table.reindex(table.shap.abs().sort_values(ascending=False).index)
    if plot:
        top = table.head(8).iloc[::-1]
        fig, ax = plt.subplots(figsize=(7, 3.2))
        ax.barh(top.checks + " = " + top.value.astype(str), top.shap,
                color=["#e34948" if s > 0 else "#2a78d6" for s in top.shap])
        ax.axvline(0, color="#52514e", lw=1)
        ax.set_xlabel("SHAP contribution (log-odds)   <- legitimate | phishing ->")
        ax.set_title(f"{verdict} ({p:.1%})", fontweight="bold")
        plt.show()
    table = table.reset_index(drop=True)
    table.attrs["probability"] = p
    return table

**A real legitimate site** (needs internet):

In [ ]:
check("https://www.wikipedia.org/").head(6)

### Test suite: fake phishing pages vs real legitimate sites

The project includes three **harmless fake phishing pages** (`api/static/demo-*.html`). They use no real brand names, and their forms send nothing. Each copies a common phishing pattern:

| Page | Pattern |
|---|---|
| `demo-phishing.html` | old-style kit: "verify your account" form submitting to `about:blank`, links that go nowhere, external scripts, hidden iframe, right-click disabled |
| `demo-invoice.html` | fake invoice: card form that **e-mails** the data (`mailto:`), status-bar spoofing, almost everything loaded from other domains |
| `demo-modern.html` | **modern kit**: clean HTML, everything on one domain, valid HTTPS. Included on purpose to show the model's main limitation. |

The fake pages are scored at their local demo address and at realistic, **fictional** phishing-style URLs (these hosts don't exist; the page's HTML is supplied directly). The legitimate sites are fetched live.

In [ ]:
def page(name):
    return open(f"api/static/{name}", encoding="utf-8").read()

TESTS = [
    # (case, url, html file or None for a live fetch, true label)
    ("Old-style phishing kit",               "http://127.0.0.1:8000/demo/account-verify", "demo-phishing.html", "phishing"),
    ("Fake invoice (form e-mails card data)", "http://127.0.0.1:8000/demo/invoice",       "demo-invoice.html",  "phishing"),
    ("Fake invoice on a phishing-style URL",
     "http://examplebank-billing.secure-update.com/invoice/88421/confirm-payment?session=7f3a9c2e1b",
                                                                                         "demo-invoice.html",  "phishing"),
    ("Modern clean phishing kit (HTTPS)",    "https://examplebank-login.com/signin",      "demo-modern.html",   "phishing"),
    ("Wikipedia",                            "https://www.wikipedia.org/",                None,                 "legitimate"),
    ("GitHub login",                         "https://github.com/login",                  None,                 "legitimate"),
    ("Python.org",                           "https://www.python.org/",                   None,                 "legitimate"),
    ("BBC",                                  "https://www.bbc.co.uk/",                    None,                 "legitimate"),
]

rows = []
for case, url, html_file, truth in TESTS:
    t = check(url, html=page(html_file) if html_file else None, plot=False)
    p = t.attrs["probability"]
    pred = "phishing" if p >= 0.5 else "legitimate"
    rows.append({"case": case, "true label": truth, "predicted": pred,
                 "phishing probability": p, "correct": "yes" if pred == truth else "NO"})
    print()
results = pd.DataFrame(rows)
print(f"Correct: {(results.correct == 'yes').sum()} of {len(results)}")
results["phishing probability"] = results["phishing probability"].map("{:.2%}".format)
results

**Why each fake page is (or isn't) caught.** Look at the SHAP explanations:

In [ ]:
check("http://127.0.0.1:8000/demo/account-verify", html=page("demo-phishing.html")).head(5)

In [ ]:
check("https://examplebank-login.com/signin", html=page("demo-modern.html")).head(5)

The **modern kit is missed**. Its HTML is clean, its links stay on its own domain, and it has valid HTTPS: exactly the signs this 2015 dataset associates with legitimate sites. Today, most phishing sites have free valid certificates and well-built pages, so a model like this needs fresher training data and reputation signals (domain age, traffic rank, blocklists) to catch them. This is the main limitation discussed in the report.

**Try your own URL.** Change it and run the cell. ⚠️ Don't paste real phishing links into a browser; checking them here only downloads their HTML and never runs their code.

In [ ]:
url = "https://github.com/login"
check(url).head(6)

## Summary

- **Best models (full set):** Random Forest, XGBoost and SVM all reach about **97.6%** test accuracy; the Decision Tree reaches 96.7%.
- **Lite (deployable) set:** about **94.6-94.8%**. The gap comes from site-reputation features (traffic rank, backlinks, domain age) that cannot be computed live.
- **Served model: XGBoost.** It has the best phishing recall and ROC-AUC, holds up best without duplicate rows, and is fast and explainable.
- **Most important features:** valid HTTPS (`sslfinal_state`) and links that point nowhere or off-site (`url_of_anchor`).
- **Limitations:** the dataset is from about 2015. Today most phishing sites have valid HTTPS, so real-world accuracy will be lower than the test accuracy. See `report.md`.